In [1]:
import json
import shutil
import duckdb
import pandas as pd
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB WAREHOUSE
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_DIR = BASE_DIR / "data_sets"
RESULT_DIR = BASE_DIR / "results"
ANALYSIS_DIR = BASE_DIR / "analysis"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = RESULT_DIR / "step03_01_star_schema.duckdb"
if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

# Load exact Step 2.3 thresholds & Step 2.4 / 2.5 expected counts
with open(RESULT_DIR / "step02_03_ETL_FINDINGS.json", "r") as f:
    step23_findings = json.load(f)

with open(RESULT_DIR / "step02_04_consistency_FINDINGS.json", "r") as f:
    step24_findings = json.load(f)

with open(RESULT_DIR / "step02_05_alignment_FINDINGS.json", "r") as f:
    step25_findings = json.load(f)

THRESH_2SD = float(step23_findings["step3_outlier_boundaries"]["global_cleaned"]["outlier_threshold_2sd"])
THRESH_3SD = float(step23_findings["step3_outlier_boundaries"]["global_cleaned"]["outlier_threshold_3sd"])
EXPECTED_BOTTLENECK_ROWS = int(step23_findings["step3_outlier_boundaries"]["global_cleaned"]["high_risk_records_gt_2sd"])
EXPECTED_CRITICAL_ROWS = int(step23_findings["step3_outlier_boundaries"]["global_cleaned"]["critical_records_gt_3sd"])
EXPECTED_TOTAL_ROWS = int(step24_findings["row_counts"]["delay_cleaned_rows"])
EXPECTED_IMPUTED_ROWS = int(step24_findings["check4_imputed_row_reconciliation"]["cleaned_imputed_rows"])
EXPECTED_ALIGNED_ROWS = int(step25_findings["step3_alignment_resolution"]["aligned_rows"])

print("=== STEP 2.3 / 2.4 / 2.5 BASELINE PARAMETERS ===")
print(f"Expected Total Rows      : {EXPECTED_TOTAL_ROWS:,}")
print(f"Expected Aligned Rows    : {EXPECTED_ALIGNED_ROWS:,}")
print(f"Expected Imputed Rows    : {EXPECTED_IMPUTED_ROWS:,}")
print(f"2 SD High-Risk Threshold : {THRESH_2SD:.6f} min")
print(f"3 SD Critical Threshold  : {THRESH_3SD:.6f} min")
print(f"Expected >2 SD Rows      : {EXPECTED_BOTTLENECK_ROWS:,}")
print(f"Expected >3 SD Rows      : {EXPECTED_CRITICAL_ROWS:,}")

=== STEP 2.3 / 2.4 / 2.5 BASELINE PARAMETERS ===
Expected Total Rows      : 36,614,824
Expected Aligned Rows    : 35,756,245
Expected Imputed Rows    : 1,032,506
2 SD High-Risk Threshold : 167.653054 min
3 SD Critical Threshold  : 233.679870 min
Expected >2 SD Rows      : 1,273,458
Expected >3 SD Rows      : 659,943


In [2]:
# ==========================================
# STEP 1: BUILD & EXPORT DIMENSION TABLES
# ==========================================
print("\n=== STEP 1: LOADING & STANDARDIZING DIMENSION TABLES ===")

dim_train_src = RESULT_DIR / "dim_train_resolved.csv"
dim_station_src = RESULT_DIR / "dim_station_resolved.csv"
dim_date_src = RESULT_DIR / "dim_date.csv"

dim_train_dst = RESULT_DIR / "dim_train.csv"
dim_station_dst = RESULT_DIR / "dim_station.csv"

# Copy resolved dimensions to canonical dim_train.csv and dim_station.csv
shutil.copyfile(dim_train_src, dim_train_dst)
shutil.copyfile(dim_station_src, dim_station_dst)

con.execute(f"""
CREATE OR REPLACE TABLE dim_train AS
SELECT 
    CAST(train_no AS INTEGER) AS train_no,
    CAST(train_name AS VARCHAR) AS train_name,
    CAST(primary_type_code AS VARCHAR) AS primary_type_code,
    CAST(is_premium_special AS BOOLEAN) AS is_premium_special,
    CAST(n_names_raw AS INTEGER) AS n_names_raw,
    CAST(raw_type_codes AS VARCHAR) AS raw_type_codes,
    CAST(raw_row_count AS INTEGER) AS raw_row_count
FROM read_csv_auto('{dim_train_dst.as_posix()}', header=True);

CREATE OR REPLACE TABLE dim_station AS
SELECT 
    CAST(station_name AS VARCHAR) AS station_name,
    CAST(station_full_name AS VARCHAR) AS station_full_name,
    CAST(station_zone AS VARCHAR) AS station_zone,
    CAST(station_address AS VARCHAR) AS station_address,
    CAST(is_address_imputed AS BOOLEAN) AS is_address_imputed
FROM read_csv_auto('{dim_station_dst.as_posix()}', header=True);

CREATE OR REPLACE TABLE dim_date AS
SELECT 
    CAST(date_id AS DATE) AS date_id,
    CAST(year AS INTEGER) AS year,
    CAST(quarter AS INTEGER) AS quarter,
    CAST(month AS INTEGER) AS month,
    CAST(month_name AS VARCHAR) AS month_name,
    CAST(day_of_month AS INTEGER) AS day_of_month,
    CAST(day_name AS VARCHAR) AS day_name,
    CAST(day_of_week AS INTEGER) AS day_of_week,
    CAST(is_weekend AS BOOLEAN) AS is_weekend
FROM read_csv_auto('{dim_date_src.as_posix()}', header=True);
""")

dim_counts = q("""
SELECT 
    (SELECT COUNT(*) FROM dim_train) AS dim_train_rows,
    (SELECT COUNT(DISTINCT train_no) FROM dim_train) AS dim_train_unique_keys,
    (SELECT COUNT(*) FROM dim_station) AS dim_station_rows,
    (SELECT COUNT(DISTINCT station_name) FROM dim_station) AS dim_station_unique_keys,
    (SELECT COUNT(*) FROM dim_date) AS dim_date_rows,
    (SELECT COUNT(DISTINCT date_id) FROM dim_date) AS dim_date_unique_keys;
""")
print(dim_counts.to_string(index=False))
FINDINGS["dimensions_summary"] = dim_counts.iloc[0].to_dict()


=== STEP 1: LOADING & STANDARDIZING DIMENSION TABLES ===
 dim_train_rows  dim_train_unique_keys  dim_station_rows  dim_station_unique_keys  dim_date_rows  dim_date_unique_keys
           8720                   8720              9027                     9027            365                   365


In [4]:
# ==========================================
# STEP 2: ENRICH SCHEDULE FEATURES & ALIGNMENT MAP
# ==========================================
print("\n=== STEP 2: PREPARING SCHEDULE FEATURES & ALIGNMENT MAP ===")

sched_path = RESULT_DIR / "combined_schedule_imputed.csv"
map_path = RESULT_DIR / "stop_alignment_map.csv"

# Precompute cumulative schedule minutes, scheduled halt duration, inter-station distance,
# and Scheduled travel time from origin for each (train_no, station_no)
con.execute(f"""
CREATE OR REPLACE TEMP TABLE schedule_enriched AS
WITH RawSched AS (
    SELECT 
        CAST(train_no AS INTEGER) AS train_no,
        CAST(station_no AS INTEGER) AS sched_station_no,
        CAST(station_name AS VARCHAR) AS sched_station_name,
        CAST(distance_from_origin AS DOUBLE) AS distance_from_origin,
        CAST(arrival_day AS INTEGER) AS arrival_day,
        CAST(arrival_time AS VARCHAR) AS arrival_time,
        CAST(departure_day AS INTEGER) AS departure_day,
        CAST(departure_time AS VARCHAR) AS departure_time,
        CAST(is_arrival_imputed AS BOOLEAN) AS is_arrival_imputed,
        CAST(is_departure_imputed AS BOOLEAN) AS is_departure_imputed,
        CAST(is_imputed AS BOOLEAN) AS is_schedule_time_imputed,
        ((CAST(arrival_day AS INTEGER) - 1) * 1440)
            + (CAST(SPLIT_PART(CAST(arrival_time AS VARCHAR), ':', 1) AS INTEGER) * 60)
            + CAST(SPLIT_PART(CAST(arrival_time AS VARCHAR), ':', 2) AS INTEGER) AS scheduled_arrival_min,
        ((CAST(departure_day AS INTEGER) - 1) * 1440)
            + (CAST(SPLIT_PART(CAST(departure_time AS VARCHAR), ':', 1) AS INTEGER) * 60)
            + CAST(SPLIT_PART(CAST(departure_time AS VARCHAR), ':', 2) AS INTEGER) AS scheduled_departure_min
    FROM read_csv_auto('{sched_path.as_posix()}', header=True)
)
SELECT 
    train_no,
    sched_station_no,
    distance_from_origin,
    COALESCE(
        distance_from_origin - LAG(distance_from_origin) OVER (PARTITION BY train_no ORDER BY sched_station_no),
        0.0
    ) AS inter_station_distance,
    arrival_day,
    arrival_time,
    departure_day,
    departure_time,
    scheduled_arrival_min,
    scheduled_departure_min,
    GREATEST(scheduled_departure_min - scheduled_arrival_min, 0) AS scheduled_halt_minutes,
    scheduled_arrival_min - FIRST_VALUE(scheduled_departure_min) OVER (
        PARTITION BY train_no ORDER BY sched_station_no
    ) AS scheduled_travel_min_from_origin,
    is_schedule_time_imputed
FROM RawSched;

CREATE OR REPLACE TEMP TABLE alignment_map AS
SELECT 
    CAST(train_no AS INTEGER) AS train_no,
    CAST(delay_station_no AS INTEGER) AS delay_station_no,
    TRY_CAST(sched_station_no AS INTEGER) AS sched_station_no,
    CAST(cls AS VARCHAR) AS alignment_cls
FROM read_csv_auto('{map_path.as_posix()}', header=True);
""")

sched_sample = q("SELECT * FROM schedule_enriched LIMIT 5;")
print(sched_sample.to_string(index=False))


=== STEP 2: PREPARING SCHEDULE FEATURES & ALIGNMENT MAP ===
 train_no  sched_station_no  distance_from_origin  inter_station_distance  arrival_day arrival_time  departure_day departure_time  scheduled_arrival_min  scheduled_departure_min  scheduled_halt_minutes  scheduled_travel_min_from_origin  is_schedule_time_imputed
    13022                21                 210.0                     5.0            1     15:54:00              1       15:56:00                    954                      956                       2                               344                     False
    13022                22                 216.0                     6.0            1     16:04:00              1       16:06:00                    964                      966                       2                               354                     False
    13022                23                 225.0                     9.0            1     16:28:00              1       16:30:00                    988 

In [6]:
# ==========================================
# STEP 3: BUILD MATERIALIZED fact_delay & EXPORT PARQUET
# ==========================================
print("\n=== STEP 3: MATERIALIZING fact_delay TABLE ===")

delay_cleaned_path = RESULT_DIR / "combined_delay_cleaned.csv"

con.execute(f"""
CREATE OR REPLACE TABLE fact_delay AS
SELECT 
    CAST(d.date AS DATE) AS date_id,
    CAST(d.train_no AS INTEGER) AS train_no,
    CAST(d.station_no AS INTEGER) AS station_no,
    CAST(d.station_name AS VARCHAR) AS station_name,
    CAST(d.delay AS DOUBLE) AS delay,
    CASE 
        WHEN d.station_no = 1 THEN 'Departure Delay' 
        ELSE 'Arrival Delay' 
    END AS delay_type,
    CASE 
        WHEN d.delay <= 0 THEN 'early_or_on_time'
        WHEN d.delay <= 15 THEN 'within_15m_tolerance'
        ELSE 'delayed'
    END AS punctuality_band,
    CASE 
        WHEN d.delay > {THRESH_3SD} THEN 'critical'
        WHEN d.delay > {THRESH_2SD} THEN 'high_risk'
        ELSE 'normal'
    END AS severity,
    CAST(d.is_imputed AS BOOLEAN) AS is_imputed,
    (m.sched_station_no IS NOT NULL) AS has_schedule,
    m.alignment_cls,
    m.sched_station_no,
    s.distance_from_origin,
    s.inter_station_distance,
    s.arrival_day,
    s.arrival_time,
    s.departure_day,
    s.departure_time,
    s.scheduled_arrival_min,
    s.scheduled_departure_min,
    s.scheduled_halt_minutes,
    s.scheduled_travel_min_from_origin,
    COALESCE(s.is_schedule_time_imputed, FALSE) AS is_schedule_time_imputed
FROM read_csv_auto('{delay_cleaned_path.as_posix()}', header=True) d
LEFT JOIN alignment_map m
  ON CAST(d.train_no AS INTEGER) = m.train_no
 AND CAST(d.station_no AS INTEGER) = m.delay_station_no
LEFT JOIN schedule_enriched s
  ON m.train_no = s.train_no
 AND m.sched_station_no = s.sched_station_no;
""")

# Clean up temporary staging tables before exporting Parquet
con.execute("DROP TABLE IF EXISTS schedule_enriched;")
con.execute("DROP TABLE IF EXISTS alignment_map;")

parquet_path = RESULT_DIR / "fact_delay.parquet"
print(f"Exporting materialized fact_delay to {parquet_path.as_posix()} ...")
con.execute(f"COPY fact_delay TO '{parquet_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD);")

parquet_size_mb = round(parquet_path.stat().st_size / (1024 * 1024), 2)
print(f"-> Exported fact_delay.parquet ({parquet_size_mb} MB)")

fact_summary = q("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE has_schedule = TRUE) AS has_schedule_rows,
    ROUND(COUNT(*) FILTER (WHERE has_schedule = TRUE) * 100.0 / COUNT(*), 4) AS has_schedule_pct,
    COUNT(*) FILTER (WHERE has_schedule = FALSE) AS no_schedule_rows,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS imputed_rows,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS observed_rows,
    COUNT(*) FILTER (WHERE has_schedule = TRUE AND is_imputed = FALSE) AS eligible_prediction_eval_rows,
    COUNT(*) FILTER (WHERE severity = 'normal') AS normal_rows,
    COUNT(*) FILTER (WHERE severity = 'high_risk') AS high_risk_rows,
    COUNT(*) FILTER (WHERE severity = 'critical') AS critical_rows,
    COUNT(*) FILTER (WHERE severity IN ('high_risk', 'critical')) AS total_bottleneck_rows,
    COUNT(*) FILTER (WHERE punctuality_band = 'early_or_on_time') AS early_or_exact_ontime_rows,
    COUNT(*) FILTER (WHERE punctuality_band = 'within_15m_tolerance') AS within_15m_rows,
    COUNT(*) FILTER (WHERE punctuality_band = 'delayed') AS delayed_gt_15m_rows
FROM fact_delay;
""")
print("\n=== FACT_DELAY SUMMARY ===")
print(fact_summary.to_string(index=False))

FINDINGS["fact_delay_summary"] = {
    **fact_summary.iloc[0].to_dict(),
    "parquet_size_mb": parquet_size_mb,
    "threshold_2sd_minutes": THRESH_2SD,
    "threshold_3sd_minutes": THRESH_3SD
}


=== STEP 3: MATERIALIZING fact_delay TABLE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exporting materialized fact_delay to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/fact_delay.parquet ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

-> Exported fact_delay.parquet (431.97 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== FACT_DELAY SUMMARY ===
 total_rows  has_schedule_rows  has_schedule_pct  no_schedule_rows  imputed_rows  observed_rows  eligible_prediction_eval_rows  normal_rows  high_risk_rows  critical_rows  total_bottleneck_rows  early_or_exact_ontime_rows  within_15m_rows  delayed_gt_15m_rows
   36614824           35756245           97.6551            858579       1032506       35582318                       35279998     35341366          613515         659943                1273458                     5572451         12571302             18471071


In [7]:
# ==========================================
# STEP 4: CREATE olap_star_schema VIEW & VALIDATE
# ==========================================
print("\n=== STEP 4: CREATING olap_star_schema VIEW & RUNNING 7 AUTOMATED CHECKS ===")

con.execute("""
CREATE OR REPLACE VIEW olap_star_schema AS
SELECT 
    f.date_id,
    dt.year,
    dt.quarter,
    dt.month,
    dt.month_name,
    dt.day_of_month,
    dt.day_name,
    dt.day_of_week,
    dt.is_weekend,
    
    f.train_no,
    tr.train_name,
    tr.primary_type_code,
    tr.is_premium_special,
    
    f.station_no,
    f.station_name,
    st.station_full_name,
    st.station_zone,
    st.station_address,
    st.is_address_imputed,
    
    f.delay,
    f.delay_type,
    f.punctuality_band,
    f.severity,
    f.is_imputed,
    
    f.has_schedule,
    f.alignment_cls,
    f.sched_station_no,
    f.distance_from_origin,
    f.inter_station_distance,
    f.arrival_day,
    f.arrival_time,
    f.departure_day,
    f.departure_time,
    f.scheduled_arrival_min,
    f.scheduled_departure_min,
    f.scheduled_halt_minutes,
    f.scheduled_travel_min_from_origin,
    f.is_schedule_time_imputed
FROM fact_delay f
LEFT JOIN dim_date dt ON f.date_id = dt.date_id
LEFT JOIN dim_train tr ON f.train_no = tr.train_no
LEFT JOIN dim_station st ON f.station_name = st.station_name;
""")

validation_df = q("""
SELECT 
    (SELECT COUNT(*) FROM fact_delay) AS fact_rows,
    (SELECT COUNT(*) FROM olap_star_schema) AS star_view_rows,
    (SELECT COUNT(*) FROM fact_delay WHERE has_schedule = TRUE) AS schedule_covered_rows,
    (SELECT COUNT(*) FROM fact_delay WHERE is_imputed = TRUE) AS imputed_rows,
    (SELECT COUNT(*) FROM fact_delay WHERE has_schedule = TRUE AND distance_from_origin IS NULL) AS missing_distance_when_has_schedule,
    (SELECT COUNT(*) FROM fact_delay f LEFT JOIN dim_station s ON f.station_name = s.station_name WHERE s.station_name IS NULL) AS unmatched_stations,
    (SELECT COUNT(*) FROM fact_delay f LEFT JOIN dim_train t ON f.train_no = t.train_no WHERE t.train_no IS NULL) AS unmatched_trains,
    (SELECT COUNT(*) FROM fact_delay f LEFT JOIN dim_date d ON f.date_id = d.date_id WHERE d.date_id IS NULL) AS unmatched_dates,
    (SELECT COUNT(*) FROM fact_delay WHERE severity IN ('high_risk', 'critical')) AS bottleneck_rows_check
""")
print(validation_df.to_string(index=False))

v = validation_df.iloc[0]
checks = {
    "check1_row_conservation": bool(v["fact_rows"] == EXPECTED_TOTAL_ROWS and v["star_view_rows"] == EXPECTED_TOTAL_ROWS),
    "check2_schedule_coverage_matches_step25": bool(v["schedule_covered_rows"] == EXPECTED_ALIGNED_ROWS),
    "check3_imputed_rows_match_step24": bool(v["imputed_rows"] == EXPECTED_IMPUTED_ROWS),
    "check4_distance_present_for_all_has_schedule": bool(v["missing_distance_when_has_schedule"] == 0),
    "check5_zero_unmatched_stations": bool(v["unmatched_stations"] == 0),
    "check6_zero_unmatched_trains": bool(v["unmatched_trains"] == 0),
    "check7_zero_unmatched_dates": bool(v["unmatched_dates"] == 0),
    "check_bottleneck_count_matches_step23": bool(v["bottleneck_rows_check"] == EXPECTED_BOTTLENECK_ROWS)
}

print("\n=== AUTOMATED VALIDATION VERDICT ===")
for k, passed in checks.items():
    print(f"  {k}: {'PASSED' if passed else 'FAILED'}")

assert all(checks.values()), "One or more Star Schema validation checks failed!"

FINDINGS["validation_checks"] = {
    **v.to_dict(),
    **checks,
    "all_checks_passed": all(checks.values())
}


=== STEP 4: CREATING olap_star_schema VIEW & RUNNING 7 AUTOMATED CHECKS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 fact_rows  star_view_rows  schedule_covered_rows  imputed_rows  missing_distance_when_has_schedule  unmatched_stations  unmatched_trains  unmatched_dates  bottleneck_rows_check
  36614824        36614824               35756245       1032506                                   0                   0                 0                0                1273458

=== AUTOMATED VALIDATION VERDICT ===
  check1_row_conservation: PASSED
  check2_schedule_coverage_matches_step25: PASSED
  check3_imputed_rows_match_step24: PASSED
  check4_distance_present_for_all_has_schedule: PASSED
  check5_zero_unmatched_stations: PASSED
  check6_zero_unmatched_trains: PASSED
  check7_zero_unmatched_dates: PASSED
  check_bottleneck_count_matches_step23: PASSED


In [8]:
# ==========================================
# STEP 5: SAMPLE OLAP ROLL-UP & EXPORT FINDINGS
# ==========================================
print("\n=== STEP 5: SAMPLE OLAP ROLL-UP BY TRAIN TYPE ===")

olap_test_df = q("""
SELECT 
    primary_type_code,
    COUNT(*) AS total_stops,
    ROUND(AVG(delay), 2) AS avg_delay_min,
    ROUND(COUNT(*) FILTER (WHERE delay <= 15) * 100.0 / COUNT(*), 2) AS on_time_15m_pct,
    ROUND(COUNT(*) FILTER (WHERE severity IN ('high_risk', 'critical')) * 100.0 / COUNT(*), 4) AS high_risk_plus_critical_pct,
    ROUND(COUNT(*) FILTER (WHERE severity = 'critical') * 100.0 / COUNT(*), 4) AS critical_pct
FROM olap_star_schema
GROUP BY primary_type_code
ORDER BY total_stops DESC;
""")
print(olap_test_df.to_string(index=False))

FINDINGS["olap_rollup_by_train_type"] = olap_test_df.to_dict(orient="records")

findings_file = RESULT_DIR / "step03_01_STAR_SCHEMA_FINDINGS.json"
with open(findings_file, "w") as f:
    json.dump(FINDINGS, f, indent=4)
print(f"\n-> Saved findings to {findings_file.as_posix()}")

con.close()


=== STEP 5: SAMPLE OLAP ROLL-UP BY TRAIN TYPE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

primary_type_code  total_stops  avg_delay_min  on_time_15m_pct  high_risk_plus_critical_pct  critical_pct
      PASS-TRAINS     17743791          32.29            51.76                       2.5022        0.9981
       EXP-TRAINS     13177628          37.03            46.56                       3.7463        2.0430
        SF-TRAINS      4888438          43.53            47.23                       5.9833        3.7252
       T18-TRAINS       338666          14.36            73.51                       0.9673        0.4928
       PRM-TRAINS       155205          96.08            45.25                      17.4183       13.5885
       RAJ-TRAINS       114441          31.56            63.69                       4.0763        2.7228
       SHT-TRAINS        98842          14.66            76.40                       1.0896        0.6060
       GRB-TRAINS        97813          50.97            48.69                       7.4101        5.1598

-> Saved findings to D:/M_Tech_CSE_AI/SEM01/D